# Workshop 2

In [ ]:
!pip -q install tiktoken
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0);

## 1 · Tokenization: text → integer ids

**Character-level** (nanoGPT's `shakespeare_char`): one token per character. The vocabulary is tiny and the sequences are long.

In [ ]:
text = "First Citizen:\nBefore we proceed any further, hear me speak.\n"
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
print("vocab size:", len(chars), "| 'hear me' ->", [stoi[c] for c in "hear me"])

**BPE (Sennrich et al., 2016):** start from characters and repeatedly **merge the most frequent adjacent pair**. The list of merges *is* the tokenizer. Toy corpus from the paper (word → frequency):

In [ ]:
from collections import Counter
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
vocab = {tuple(w) + ("</w>",): f for w, f in corpus.items()}

def merge_word(syms, pair):
    out, i = [], 0
    while i < len(syms):
        if i < len(syms) - 1 and (syms[i], syms[i+1]) == pair:
            out.append(syms[i] + syms[i+1]); i += 2
        else:
            out.append(syms[i]); i += 1
    return tuple(out)

merges = []
for step in range(8):
    pairs = Counter()
    for syms, f in vocab.items():
        for p in zip(syms, syms[1:]):
            pairs[p] += f
    best = max(pairs, key=pairs.get)
    merges.append(best)
    vocab = {merge_word(s, best): f for s, f in vocab.items()}
    print(f"merge {step+1}: {best[0]} + {best[1]}  ->  {''.join(best)}  ({pairs[best]}x)")

def bpe(word):                       # apply the merges in order to a NEW word
    syms = tuple(word) + ("</w>",)
    for m in merges:
        syms = merge_word(syms, m)
    return syms

print("\n'lowest' (never seen) ->", bpe("lowest"))

**GPT-2's BPE** (used in Workshop 3): it works on **bytes**, so no input is ever unknown, and it has 50,000 merges. A leading space is part of the token.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print("vocab size:", enc.n_vocab)
for s in ["Hello world! Tokenization is fun.", "La tokenizzazione è divertente.", "12345 + 67890"]:
    ids = enc.encode(s)
    print(f"{len(ids):2d} tokens:", [enc.decode([i]) for i in ids])

## 2 · Embedding layer (`wte`): ids → vectors

`nn.Embedding(V, d)` is a **lookup table**: token `i` → row `i` of a `(V, d)` matrix W.
**Link to Workshop 1:** this is the same as a linear layer (no bias) applied to a one-hot vector, because $e_i^\top W$ = row $i$ of $W$. The lookup just skips multiplying by all those zeros.

In [ ]:
V, d = 10, 4
wte = nn.Embedding(V, d)
idx = torch.tensor([3, 7, 3])

one_hot = F.one_hot(idx, V).float()                 # (3, V)
lin = nn.Linear(V, d, bias=False)
lin.weight.data = wte.weight.data.T.clone()         # nn.Linear stores W transposed

print("wte(idx):\n", wte(idx).data)
print("lookup == one_hot @ W :", torch.allclose(wte(idx), one_hot @ wte.weight))
print("lookup == nn.Linear   :", torch.allclose(wte(idx), lin(one_hot)))

In GPT-2 small, `wte` is 50304 × 768 ≈ 38.6M parameters. nanoGPT shares it with the output layer (`lm_head`), a trick called *weight tying*.

## 3 · Positional encoding

**Why:** attention has no notion of order. Shuffle the input and the output is shuffled the same way, nothing else changes.

In [ ]:
x, perm = torch.randn(1, 5, 8), torch.tensor([3, 0, 4, 1, 2])
attn = lambda x: F.scaled_dot_product_attention(x, x, x)
print("attn(shuffled x) == shuffled attn(x):", torch.allclose(attn(x[:, perm]), attn(x)[:, perm], atol=1e-6))

### Main case: learned absolute positions (GPT-2 / nanoGPT `wpe`)
A second lookup table, indexed by **position** (0, 1, 2, …) and **added** to the token embeddings. It's learned by backprop like any other weight.

In [ ]:
block_size, n_embd = 8, 16
wte = nn.Embedding(enc.n_vocab, n_embd)
wpe = nn.Embedding(block_size, n_embd)

def embed(s):                                         # as in nanoGPT's forward
    idx = torch.tensor([enc.encode(s)])
    t = idx.shape[1]
    assert t <= block_size, "no row in wpe for that position!"
    pos = torch.arange(0, t)
    return wte(idx) + wpe(pos)                        # (1, T, n_embd)

a, b = embed(" dog bites man"), embed(" man bites dog")
print(tuple(a.shape))
print("' dog' at position 0 vs position 2 identical?", torch.allclose(a[0, 0], b[0, 2]))

The limitation: the table has exactly `block_size` rows (1024 in GPT-2), so it can't handle a longer sequence.

### Sinusoidal PE (original Transformer, 2017)
A **fixed formula** with no parameters: each pair of dimensions is a sin/cos "clock" at a different frequency. Also **added** to the token embeddings, and it works for any position.

In [ ]:
T, D = 256, 64
pos, freq = torch.arange(T)[:, None], 10000 ** (-torch.arange(0, D, 2) / D)
pe = torch.zeros(T, D); pe[:, 0::2] = torch.sin(pos * freq); pe[:, 1::2] = torch.cos(pos * freq)
plt.figure(figsize=(9, 2.5)); plt.imshow(pe.T, aspect="auto", cmap="RdBu")
plt.xlabel("position"); plt.ylabel("dim"); plt.title("sinusoidal positional encoding"); plt.show()

### RoPE (Llama, Mistral, Qwen, …)
Nothing is added to the embeddings. Inside attention, **q and k are rotated** by an angle proportional to their position. So the score $q_m \cdot k_n$ depends only on the **distance** $m - n$:

In [ ]:
def rope(x, p):
    th = p * 10000 ** (-torch.arange(0, x.shape[-1], 2) / x.shape[-1])
    x1, x2 = x[0::2], x[1::2]
    return torch.stack([x1 * th.cos() - x2 * th.sin(), x1 * th.sin() + x2 * th.cos()], -1).flatten()

q, k = torch.randn(64), torch.randn(64)
for m, n in [(5, 2), (105, 102), (1005, 1002), (5, 4)]:
    print(f"q@{m:<4} k@{n:<4} distance {m-n}: score {rope(q, m) @ rope(k, n):.4f}")